# A personality transplant for an AI 🏴‍☠️
### 5,000 dials · a few minutes on one GPU · the model itself never changes

A language model is 500,000,000 numbers. Rewriting them all (a "full fine-tune") is how you
normally change a model's behavior — expensive, slow, and you end up storing a whole new model.

**NTK fine-tuning** learns a tiny **controller** — 5,000 dials, a 100 KB file — that rides along
at serving time and re-balances how the model expresses itself. The original model stays untouched:
one base model, many personalities, each a 100 KB file.

To prove the point we gave our model a *pirate* personality, from just **79 example answers**
written in pirate voice. Training took a few minutes on one GPU. (The same training on a laptop
CPU: over an hour for a fraction of the work — the platform's GPU does in minutes what a desktop
can't.)


## Setup


In [ ]:
import json, re, textwrap
from pathlib import Path

import requests

# Live NTK-pirate endpoint (in-cluster service DNS).
PIRATE_ENDPOINT = "http://ntk-pirate-predictor.admin.svc.cluster.local/openai/v1"

RESULTS_DIR = Path("demo_results")
base = json.loads((RESULTS_DIR / "style_base.json").read_text())
pirate = json.loads((RESULTS_DIR / "style_ntk.json").read_text())

def served_model_name(endpoint):
    r = requests.get(endpoint.rstrip("/") + "/models", timeout=30)
    r.raise_for_status()
    return r.json()["data"][0]["id"]

def ask_live(endpoint, model, question, max_tokens=120):
    prompt = question if question.startswith("Question:") else f"Question: {question}\nAnswer:"
    body = {"model": model, "prompt": prompt, "temperature": 0,
            "max_tokens": max_tokens, "stop": ["Question:", "\n"]}
    r = requests.post(endpoint.rstrip("/") + "/completions", json=body, timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["text"].strip()

def tidy(text):
    # First line, trimmed to the last complete sentence.
    line = text.strip().splitlines()[0].strip()
    m = list(re.finditer(r"[.!?]", line))
    return line[: m[-1].end()] if m else line

def show(title, question, answer):
    q = question.removeprefix("Question:").removesuffix("Answer:").strip()
    print("=" * 78)
    print(f"{title}")
    print("-" * 78)
    print(textwrap.fill("Q: " + q, width=78))
    print()
    print(textwrap.fill("A: " + tidy(answer), width=78, subsequent_indent="   "))
    print()

# Showcase: hand-vetted items (pirate voice + correct fact); falls back to any pirate item.
SHOWCASE = [1, 13, 9]
if not all(pirate["items"][i]["pirate"] for i in SHOWCASE):
    SHOWCASE = [i for i, it in enumerate(pirate["items"]) if it["pirate"]][:3]
print(f"Loaded {base['n_items']} held-out questions. Showcase items: {SHOWCASE}")


## 1 · BEFORE — the original model

Perfectly correct. Perfectly boring. (Real captured answers, greedy decoding.)


In [ ]:
for i in SHOWCASE:
    it = base["items"][i]
    show("ORIGINAL MODEL", it["question"], it["generation"])


## 2 · AFTER — same model + a 100 KB pirate controller, answering LIVE

The model's 500,000,000 numbers are byte-for-byte identical. Only the 5,000-dial controller
was added at serving time.


In [ ]:
PIRATE_MODEL = served_model_name(PIRATE_ENDPOINT)
print(f"live endpoint model: {PIRATE_MODEL}\n")
for i in SHOWCASE:
    it = pirate["items"][i]
    live = ask_live(PIRATE_ENDPOINT, PIRATE_MODEL, it["question"])
    show("PIRATE MODEL (LIVE) 🏴\u200d☠\ufe0f", it["question"], live)


## 3 · Ask it anything (live)

Type any question — it has a personality now.


In [ ]:
YOUR_QUESTION = "What is the best way to travel?"
print(tidy(ask_live(PIRATE_ENDPOINT, PIRATE_MODEL, YOUR_QUESTION)))


## 4 · Scoreboard


In [ ]:
import math
import matplotlib.pyplot as plt

labels = ["Original\nmodel", "NTK-tuned\n(+5,000 dials)"]
colors = ["#9aa5b1", "#7b1fa2"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
rate = [base["pirate_rate"] * 100, pirate["pirate_rate"] * 100]
bars = ax1.bar(labels, rate, color=colors)
ax1.set_ylabel("% of answers in pirate voice")
ax1.set_title(f"Personality ({base['n_items']} unseen questions)")
ax1.set_ylim(0, 105)
for b, v in zip(bars, rate):
    ax1.text(b.get_x() + b.get_width() / 2, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")

ppl = [base.get("perplexity") or math.exp(base["nll"]), pirate.get("perplexity") or math.exp(pirate["nll"])]
bars2 = ax2.bar(labels, ppl, color=colors)
ax2.set_ylabel("surprise per word (lower is better)")
ax2.set_title("How surprised is the model by pirate answers?")
for b, v in zip(bars2, ppl):
    ax2.text(b.get_x() + b.get_width() / 2, v * 1.01, f"{v:.2f}", ha="center", fontweight="bold")

impr = (1 - pirate["nll"] / base["nll"]) * 100
fig.suptitle(f"One base model, one 100 KB controller — {impr:.0f}% better at the new personality",
             fontweight="bold")
fig.tight_layout()
plt.show()


## 5 · How it was done on the platform

1. **79 example answers** in the target voice, uploaded as a dataset.
2. **One training run** (a few hundred steps, under a minute of GPU compute) → a 100 KB `controller.pt` in the
   model catalog.
3. **One serving call** — the controller attaches to the untouched base model:

```json
POST /cogapi/models-serving
{
  "isvc_name": "ntk-pirate",
  "model_id":  "8f7318ab-6e4e-4e94-aeb9-6a61b6dd51e6",
  "llm_adapter": { "kind": "ntk_model",
                   "adapter_model_id": "9c2a4c6e-ea69-4c18-a69f-f88c2dc4a85b" }
}
```

**Why it matters:** the personality is a 100 KB file. A support-tone, a legal-tone, a
brand-voice — dozens of them can sit in the catalog, trained in minutes each, all sharing one
base model on one GPU.


---
## Presenter appendix (ops — not part of the audience demo)

- **One L40 GPU = one live model.** The math demo (`ntk-gsm8k`) and this demo (`ntk-pirate`)
  can't run simultaneously. Flip (takes ~3 min):
  - `kubectl scale deploy ntk-pirate-predictor -n admin --replicas=0`
  - `kubectl scale deploy ntk-gsm8k-predictor -n admin --replicas=1` (and vice versa)
  - qwen38 restore after all demos: scale both demos to 0, then
    `kubectl scale deploy qwen38-predictor -n admin --replicas=1`
- **Pre-stage check:** run Setup + `served_model_name(PIRATE_ENDPOINT)`.
- Controller row `9c2a4c6e-ea69-4c18-a69f-f88c2dc4a85b` (type `ntk_controller`, base
  Qwen2.5-0.5B-Instruct row `8f7318ab…`); trained 2026-09-03 on the cluster L40
  (`cfhfserver:0.18.11-gpu` image, `ntkmirror fit`, gates 5000 / steps 480 / lr 5e-3 / max_log_gate 0.5,
  dataset `runs/pirate/train.jsonl`, seeded generator `scripts/make_pirate_dataset.py`).
- `demo_results/style_*.json` captured in-cluster with `scripts/style_eval.py`.
- The pirate-voice % counts answers containing pirate-register markers (word-boundary matches:
  arr, matey, ye, savvy, …) — crude but audience-honest; read a few transcripts too.
- **Live-question cell: pre-test every question you plan to ask on stage.** Decoding is
  deterministic (temp 0), so a tested question always gives the same answer — but an untested one
  can produce nonsense or crude words (this is a 0.5B model with a strong style push). Facts also
  wobble under the style controller (~half the held-out answers keep the right fact) — pick
  showcase questions where fact + voice are both right, and say so if asked: the personality is
  the demo, not factual accuracy.
- Controller sweep (2026-09-03, L40, 480 steps each): max_log_gate 0.05 → NLL 3.97, zero visible
  style; 0.2 → mild style; **0.5 → served: 9/16 pirate voice, NLL 1.89 (chosen)**; 1.0 → strongest
  voice but degraded coherence. Checkpoints in `runs/pirate/controller_*.pt`; the clamp is baked
  into the checkpoint and honored by the serving plugin.
